# RiverSense
### Prognoza stanu wody Wisły w Krakowie z wykorzystaniem uczenia maszynowego

W lipcu 1997 r. Wisła w Krakowie podniosła się do 867 cm, a w maju 2010 r. – do rekordowych 902 cm.
Woda, która zalewa Kraków, nie bierze się znikąd: zanim dotrze do wodowskazu **Kraków-Bielany**, płynie
przez Beskidy, Sołą, Skawą, Przemszą i górną Wisłą. Każde wezbranie widać więc wcześniej – na stacjach
w górze rzeki i w opadach nad zlewnią. Projekt RiverSense próbuje tę wiedzę zamienić w prognozę.

**Cel:** prognoza dobowego stanu wody na Wiśle w Krakowie-Bielanach na **1, 2 i 3 dni do przodu**
i ostrzeżenie, gdy prognoza przekroczy **stan ostrzegawczy (370 cm)** lub **alarmowy (520 cm)**.

**Dane:**

| Źródło | Co mierzy | Okres | Rola |
|---|---|---|---|
| Archiwum IMGW | dobowe stany wody na 19 stacjach w zlewni, zjawiska lodowe | 1980–2025 | trening i walidacja |
| ERA5 (Open-Meteo) | godzinowy opad, śnieg, temperatura, wilgotność gleby w 7 punktach | 1980 – obecnie | trening i walidacja |
| API IMGW | bieżące stany wody, co godzinę | od 12.10.2026 | test na żywo |
| Open-Meteo | prognoza pogody na 3 dni, zapisywana co godzinę | od 12.10.2026 | test na żywo |
| IMGW | oficjalne ostrzeżenia hydrologiczne | od 12.10.2026 | porównanie z modelem |

W historii znamy faktyczny opad, ale działający system zna tylko jego prognozę. Dlatego od października
2026 r. zbieramy prognozy dokładnie w takiej postaci, w jakiej były dostępne w danej chwili – to pozwala
uczciwie ocenić model w warunkach rzeczywistych.

**Plan notatnika:**
1. Import bibliotek
2. Data preprocessing – od surowych plików do czystych tabel
3. Eksploracyjna analiza danych (EDA)
4. Cechy i model *(kolejny etap)*
5. Ewaluacja i test na danych live *(kolejny etap)*

## 1. Import bibliotek

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data import (build, combine, config, hist_hydro, hist_meteo, live_hydro,
                      live_meteo, live_warnings, quality, schema, stations, utils)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (12, 4)

print(f"pandas {pd.__version__}, numpy {np.__version__}")
print(f"Repozytorium: {ROOT}")

pandas 2.2.3, numpy 2.2.5
Repozytorium: C:\Users\Admin\RiverSense


## 2. Data preprocessing

Surowe dane pochodzą z różnych źródeł, mają różne strefy czasowe, rozdzielczości, kody braków i błędy
odczytu. Ta sekcja zamienia je w spójne tabele w `data/processed/`. Cała logika jest w module `src/data/`
(reguły: `docs/plan_czyszczenia_danych.md`) – notatnik tylko wywołuje jego funkcje, dzięki czemu trening
i system prognoz dostają dane przygotowane identycznie. Te same tabele buduje polecenie
`python -m src.data.build`.

Kolejność kroków:

| Krok | Dane | Moduł |
|---|---|---|
| 2.1 | konfiguracja i wspólne zasady | `config.py`, `schema.py` |
| 2.2 | przegląd surowych plików | – |
| 2.3 | stany wody – historia (+ zjawiska lodowe) | `hist_hydro.py` |
| 2.4 | pogoda – historia (ERA5) | `hist_meteo.py` |
| 2.5 | stany wody – live | `live_hydro.py` |
| 2.6 | prognozy pogody i synop – live | `live_meteo.py` |
| 2.7 | ostrzeżenia IMGW – live | `live_warnings.py` |
| 2.8 | łączenie, tabela stacji, walidacja, zapis | `combine.py`, `stations.py`, `build.py` |

### 2.1 Konfiguracja

Wszystkie progi i okresy są w `src/data/config.py` – zmieniamy je tylko tam.

Wspólne zasady wszystkich tabel: czas w **UTC**, `station_id` jako tekst, nazwy kolumn z jednostką
(`level_cm`, `precip_mm`…), braki zawsze jako `NaN`, flagi `is_imputed` / `is_suspect` / `is_spike`,
kolumna `source` (`hist`, `live`, `era5`, `forecast`).

In [2]:
pd.Series({
    "stacja docelowa": f"{config.TARGET} – {config.STATIONS[config.TARGET][0]}",
    "okres historii": f"{config.HIST_START} – {config.HIST_END}",
    "definicja doby (otwarta decyzja nr 1)": config.DAILY_DEFINITION,
    "max luka do interpolacji – historia": f"{config.MAX_GAP_DAYS} dni",
    "max luka do interpolacji – live": f"{config.MAX_GAP_HOURS} h",
    "przestarzały odczyt live": f"> {config.STALE_HOURS} h",
    "skok dobowy / godzinowy": f"{config.SPIKE_DAY_CM} cm / {config.SPIKE_HOUR_CM} cm",
    "zakres stanu": f"{config.LEVEL_RANGE} cm",
    "stacje bez roli głównej cechy": ", ".join(config.NOT_MAIN_FEATURE),
}, name="wartość").to_frame()

,wartość
stacja docelowa,150190340 – Kraków-Bielany
okres historii,1980-11-01 – 2025-10-31
definicja doby (otwarta decyzja nr 1),06utc
max luka do interpolacji – historia,3 dni
max luka do interpolacji – live,3 h
przestarzały odczyt live,> 3 h
skok dobowy / godzinowy,150 cm / 50 cm
zakres stanu,"(0, 1200) cm"
stacje bez roli głównej cechy,"149180210, 149190120, 149190200"


Sieć rzeczna – każda stacja wskazuje najbliższy wodowskaz w dół rzeki. Na tej podstawie sprawdzamy,
czy skok stanu widać też na stacjach sąsiednich (prawdziwe wezbranie), czy tylko na jednej (błąd odczytu).

In [3]:
pd.DataFrame([
    {"station_id": sid, "stacja": name, "rzeka": river,
     "w dół rzeki": config.STATIONS[down][0] + f" ({down})" if down else "–",
     "sąsiedzi": ", ".join(config.STATIONS[n][0] for n in utils.neighbors(sid))}
    for sid, (name, river, down) in config.STATIONS.items()
])

,station_id,stacja,rzeka,w dół rzeki,sąsiedzi
0,149180100,Skoczów,Wisła,Zabrzeg (149180210),Zabrzeg
1,149180210,Zabrzeg,Wisła,Goczałkowice (149180240),"Skoczów, Goczałkowice"
2,149180240,Goczałkowice,Wisła,Jawiszowice (149190060),"Zabrzeg, Jawiszowice"
3,149190060,Jawiszowice,Wisła,Pustynia (150190170),"Goczałkowice, Pustynia"
4,150190170,Pustynia,Wisła,Smolice (150190260),"Jawiszowice, Oświęcim, Jeleń, Smolice"
5,150190260,Smolice,Wisła,Kraków-Bielany (150190340),"Pustynia, Zator, Kraków-Bielany"
6,150190340,Kraków-Bielany,Wisła,–,"Smolice, Radziszów, Balice"
7,149190100,Żywiec,Soła,Czaniec-Kobiernice (149190120),Czaniec-Kobiernice
8,149190120,Czaniec-Kobiernice,Soła,Oświęcim (150190160),"Żywiec, Oświęcim"
9,150190160,Oświęcim,Soła,Pustynia (150190170),"Czaniec-Kobiernice, Pustynia"


### 2.2 Przegląd surowych plików

Surowe pliki są **tylko do odczytu** – nic ich nie nadpisuje, więc wynik zawsze da się odbudować od zera.

In [4]:
files = sorted([*config.HIST.glob("*.parquet"), *config.HIST.glob("*.csv"), *config.LIVE.glob("*.csv")])
pd.DataFrame([{"plik": str(f.relative_to(ROOT)), "MB": round(f.stat().st_size / 1e6, 2)} for f in files])

,plik,MB
0,data\historical\hydro_daily.parquet,1.97
1,data\historical\ice_daily.parquet,0.06
2,data\historical\meteo_hourly_katowice.parquet,4.20
3,data\historical\meteo_hourly_krakow.parquet,4.19
4,data\historical\meteo_hourly_makow_podhalanski...,4.19
5,data\historical\meteo_hourly_oswiecim.parquet,4.21
6,data\historical\meteo_hourly_wadowice.parquet,4.21
7,data\historical\meteo_hourly_wisla_beskidy.par...,4.25
8,data\historical\meteo_hourly_zywiec.parquet,4.20
9,data\historical\stations_found.csv,0.00


Dane live zapisuje `collect.py` jako tekst, z czasem IMGW **w UTC bez oznaczenia strefy** (sprawdzone na API:
o 15:15 UTC API podaje pomiar z 14:50). Wyjątek: ostrzeżenia IMGW są w czasie polskim.

In [5]:
live_hydro.load_raw().tail(3)

,station_id,station,measured_at,water_level_cm,flow_m3s,flow_at,warning_cm,alarm_cm,fetched_at
1223,150190340,Kraków-Bielany (Wisła),2026-10-15 12:50:00,147,235,2025-11-27 23:50:00,370,520,2026-10-15T13:20:47Z
1224,150190340,Kraków-Bielany (Wisła),2026-10-15 13:20:00,146,235,2025-11-27 23:50:00,370,520,2026-10-15T14:01:16Z
1225,150190340,Kraków-Bielany (Wisła),2026-10-15 14:20:00,146,235,2025-11-27 23:50:00,370,520,2026-10-15T15:01:18Z


### 2.3 Stany wody – historia

`hydro_daily.parquet` (archiwum IMGW, dobowe) + `ice_daily.parquet` (zjawiska lodowe). Kroki:

1. kontrola listy 19 stacji,
2. okres od 1.11.1980 (od tego dnia są dane z Bielan) i **pełny kalendarz** – brakujące dni jako `NaN`,
3. **lód** – zator lub pokrywa lodowa podnosi stan bez wzrostu przepływu, więc skoki w dni z lodem nie są usuwane,
4. **zakres** – stan < 0 lub > rekord stacji + 100 cm → `is_suspect`,
5. **pojedyncze skoki** > 150 cm/dobę, które wracają następnego dnia i nie występują na stacjach sąsiednich → `NaN`,
6. **zmiany zera wodowskazu** – tylko wykrycie kandydatów, decyzja w raporcie,
7. **krótkie luki** ≤ 3 dni – interpolacja liniowa (`is_imputed`), dłuższe zostają `NaN`.

In [6]:
raw_hydro, raw_ice = hist_hydro.load_raw()
print(f"surowe: {len(raw_hydro):,} wierszy, {raw_hydro['station_id'].nunique()} stacji, "
      f"{raw_hydro['date'].min():%Y-%m-%d} – {raw_hydro['date'].max():%Y-%m-%d}")
print(f"zjawiska lodowe: {len(raw_ice):,} dni")

hist_levels, hist_info = hist_hydro.process()
hist_levels.head()

surowe: 409,462 wierszy, 19 stacji, 1950-11-01 – 2025-10-31
zjawiska lodowe: 17,536 dni


,date,station_id,level_cm,flow_m3s,has_ice,is_imputed,is_suspect,is_spike,source
0,1980-11-01,149180100,147.0,3.01,False,False,False,False,hist
1,1980-11-01,149180210,NaN,NaN,False,False,False,False,hist
2,1980-11-01,149180240,101.0,4.10,False,False,False,False,hist
3,1980-11-01,149190060,270.0,6.50,False,False,False,False,hist
4,1980-11-01,149190100,211.0,7.28,False,False,False,False,hist


In [7]:
names = {sid: f"{n} ({r})" for sid, (n, r, _) in config.STATIONS.items()}
pd.DataFrame({
    "braki przed": hist_info["missing_days_raw"],
    "uzupełnione (≤ 3 dni)": hist_info["imputed_days"],
    "braki po": hist_info["missing_days_after"],
    "dni z lodem": hist_levels.groupby("station_id")["has_ice"].sum(),
}).rename(index=names)

,braki przed,uzupełnione (≤ 3 dni),braki po,dni z lodem
station_id,,,,
Skoczów (Wisła),0,0,0,1231
Zabrzeg (Wisła),11709,2,11707,4
Goczałkowice (Wisła),0,0,0,493
Jawiszowice (Wisła),0,2,0,451
Żywiec (Soła),0,0,0,1960
Czaniec-Kobiernice (Soła),12133,0,12133,4
Zator (Skawa),465,9,456,915
Wadowice (Skawa),730,0,730,1602
Sucha Beskidzka (Stryszawka),0,0,0,2177


Usunięte skoki (błędy odczytu) i kandydaci na zmianę zera wodowskazu:

In [8]:
display(hist_info["spikes"])
hist_info["zero_shifts"].assign(stacja=lambda d: d["station_id"].map(names))

,date,station_id,level_cm
0,1989-07-10,149190060,460.0
1,2002-05-30,149190060,553.0


,station_id,hydro_year,min_before_cm,min_after_cm,shift_cm,stacja
0,149190120,2022,186.0,147.0,-39.0,Czaniec-Kobiernice (Soła)
1,149190180,1987,112.0,78.0,-34.0,Wadowice (Skawa)
2,149190180,1992,70.0,32.0,-38.0,Wadowice (Skawa)
3,149190180,1996,22.0,70.0,48.0,Wadowice (Skawa)
4,149190180,1999,70.0,18.0,-52.0,Wadowice (Skawa)
5,149190180,2009,11.0,107.0,96.0,Wadowice (Skawa)
6,149190210,2010,104.0,73.0,-31.0,Sucha Beskidzka (Skawa)
7,149190260,2001,164.0,128.0,-36.0,Osielec (Skawa)
8,149190260,2010,110.0,75.0,-35.0,Osielec (Skawa)
9,149190260,2021,85.0,45.0,-40.0,Osielec (Skawa)


Kandydaci na zmianę zera wodowskazu to trwałe przesunięcia rocznego minimum o > 30 cm. Mogą wynikać ze zmiany
zera, ale też z erozji koryta lub pracy zbiornika (np. Świnna Poręba powyżej Wadowic). Niczego nie
korygujemy automatycznie – to decyzja zespołu na podstawie EDA.

Szybka kontrola zdrowego rozsądku – maksimum na Bielanach powinno przypadać na powódź z maja 2010:

In [9]:
bielany = hist_levels[hist_levels["station_id"] == config.TARGET].set_index("date")["level_cm"]
print(f"maksimum: {bielany.max():.0f} cm, {bielany.idxmax():%Y-%m-%d}")
print(f"dni ze stanem ≥ ostrzegawczy (370 cm): {(bielany >= 370).sum()}, ≥ alarmowy (520 cm): {(bielany >= 520).sum()}")

maksimum: 902 cm, 2010-05-18
dni ze stanem ≥ ostrzegawczy (370 cm): 158, ≥ alarmowy (520 cm): 38


### 2.4 Pogoda – historia (ERA5)

7 punktów w zlewni, dane godzinowe od 1980 r. Kroki: jedna tabela z kolumną `point`, czas UTC, pełna siatka
godzinowa, nazwy z jednostką, pokrywa śnieżna z m na cm, ujemny opad → 0, zakresy fizyczne → `is_suspect`.

**Znacznik czasu = koniec godziny:** opad o 01:00 to suma z 00:00–01:00.

In [10]:
era5, era5_info = hist_meteo.process()
print(era5_info)
era5.head()

{'era5_end': Timestamp('2026-10-09 23:00:00+0000', tz='UTC'), 'missing_hours': 0, 'nan_values': 0, 'suspect_hours': 0}


,point,time_utc,precip_mm,rain_mm,snowfall_cm,snow_depth_cm,temp_c,soil_moist_m3m3,is_suspect,source
0,katowice,1980-01-01 00:00:00+00:00,0.0,0.0,0.0,3.0,-5.8,0.361,False,era5
1,katowice,1980-01-01 01:00:00+00:00,0.0,0.0,0.0,3.0,-4.5,0.360,False,era5
2,katowice,1980-01-01 02:00:00+00:00,0.0,0.0,0.0,3.0,-3.6,0.360,False,era5
3,katowice,1980-01-01 03:00:00+00:00,0.0,0.0,0.0,3.0,-3.1,0.360,False,era5
4,katowice,1980-01-01 04:00:00+00:00,0.0,0.0,0.0,3.0,-3.3,0.360,False,era5


**Agregacja dobowa** jest zgodna z definicją doby stanu wody (`config.DAILY_DEFINITION`). Przy odczycie stanu
o 06 UTC doba opadu D to godziny (D−1 06:00, D 06:00] UTC – opad, który spadł przed odczytem stanu.
Sumy liczymy tylko z pełnych dób (24 godziny), pokrywa śnieżna to wartość o 06 UTC.

Kontrola: suma dobowa opadu w Krakowie 16.05.2010 musi być równa sumie godzin z okna tej doby.

In [11]:
era5_daily = hist_meteo.aggregate_daily(era5)

day = pd.Timestamp("2010-05-16")
window = era5[(era5["point"] == "krakow")
              & (era5["time_utc"] > (day - pd.Timedelta(hours=18)).tz_localize("UTC"))
              & (era5["time_utc"] <= (day + pd.Timedelta(hours=6)).tz_localize("UTC"))]
print(f"suma godzin: {window['precip_mm'].sum():.1f} mm, wartość dobowa: "
      f"{era5_daily.set_index(['point', 'date']).at[('krakow', day), 'precip_mm']:.1f} mm")
era5_daily.tail()

suma godzin: 19.9 mm, wartość dobowa: 19.9 mm


,point,date,precip_mm,rain_mm,snowfall_cm,temp_mean_c,temp_min_c,temp_max_c,soil_moist_m3m3,n_hours,source,snow_depth_cm
119590,zywiec,2026-10-06,0.0,0.0,0.0,11.745833,5.4,19.500000,0.303042,24,era5,0.0
119591,zywiec,2026-10-07,0.0,0.0,0.0,12.041667,4.0,21.500000,0.298542,24,era5,0.0
119592,zywiec,2026-10-08,0.0,0.0,0.0,14.608334,10.1,21.700001,0.292667,24,era5,0.0
119593,zywiec,2026-10-09,8.7,8.7,0.0,15.266666,9.2,20.400000,0.323375,24,era5,0.0
119594,zywiec,2026-10-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17,era5,NaN


### 2.5 Stany wody – live

`hydro.csv` – pomiary IMGW pobierane co godzinę przez GitHub Actions. Kroki:

1. typy (wszystko było tekstem) i czas UTC,
2. **przestarzałe odczyty** – pomiar starszy od pobrania o > 3 h (stacja nie przysłała nowego) jest odrzucany,
3. przepływ tylko, gdy młodszy niż 24 h (IMGW aktualizuje go rzadko),
4. **siatka godzinowa** – ostatni pomiar w każdej godzinie (15:00 = ostatni pomiar z 14:00–15:00),
5. skoki > 50 cm/h bez potwierdzenia na sąsiednich stacjach → `NaN`, luki ≤ 3 h – interpolacja,
6. **wartość dobowa** liczona tak jak IMGW (`config.DAILY_DEFINITION`), żeby live było porównywalne z historią.

In [12]:
live_hourly, live_info = live_hydro.process()
print(f"wiersze surowe: {live_info['raw_rows']}, duplikaty: {live_info['duplicates']}")
print(f"mediana opóźnienia pomiaru względem pobrania: {live_info['lag_median_h']:.2f} h")
print(f"przestarzałe odczyty (odrzucone): {live_info['stale_rows'].rename(index=names).to_dict()}")
print(f"siatka: {live_hourly['time_utc'].min()} – {live_hourly['time_utc'].max()}")
pd.DataFrame({"brak godzin (surowe)": live_info["missing_hours_raw"],
              "uzupełnione": live_info["imputed_hours"],
              "brak po": live_info["missing_hours_after"]}).rename(index=names)

wiersze surowe: 1226, duplikaty: 0
mediana opóźnienia pomiaru względem pobrania: 0.69 h
przestarzałe odczyty (odrzucone): {'Sucha Beskidzka (Stryszawka)': 1}
siatka: 2026-10-12 18:00:00+00:00 – 2026-10-15 15:00:00+00:00


,brak godzin (surowe),uzupełnione,brak po
station_id,,,
Skoczów (Wisła),17,2,15
Zabrzeg (Wisła),3,3,0
Goczałkowice (Wisła),17,2,15
Jawiszowice (Wisła),3,3,0
Żywiec (Soła),3,3,0
Czaniec-Kobiernice (Soła),3,3,0
Zator (Skawa),17,2,15
Wadowice (Skawa),17,2,15
Sucha Beskidzka (Stryszawka),70,0,70


Braki na początku szeregu u części stacji to nie błąd – 8 stacji dodano do zbierania dopiero 13.10 o 09:00.
Stryszawka nie ma żadnych aktualnych odczytów (od czerwca API zwraca ten sam stary pomiar).

In [13]:
live_daily = live_hydro.aggregate_daily(live_hourly)
live_daily[live_daily["station_id"] == config.TARGET]

,station_id,date,level_cm,flow_m3s,has_ice,is_imputed,is_suspect,is_spike,source
246,150190340,2026-10-13,148.0,NaN,False,False,False,False,live
702,150190340,2026-10-14,147.0,NaN,False,False,False,False,live
1158,150190340,2026-10-15,148.0,NaN,False,False,False,False,live


### 2.6 Prognozy pogody i synop – live

`forecast_RRRR-MM.csv` – co godzinę zapisywana prognoza Open-Meteo na 3 dni (+ 1 dzień wstecz) dla tych samych
7 punktów co w historii. Kolumna `lead_h = valid_at − issued_at`: ujemne to analiza modelu, dodatnie – prognoza.

Wynikowe tabele:
- `forecast_archive` – wszystkie prognozy (ocena trafności, test na żywo),
- `forecast_latest` – dla każdej godziny najnowsza prognoza wydana **przed** nią (to, co model naprawdę wiedział),
- analiza z prognozy (`lead_h` −48…0) uzupełnia pogodę „obserwowaną” w godzinach, dla których nie ma jeszcze ERA5
  (opóźnienie ok. 6 dni).

In [14]:
forecast, forecast_info = live_meteo.process_forecasts()
print(forecast_info)
forecast_latest = live_meteo.latest(forecast)
observed = live_meteo.observed_proxy(forecast)
print(f"forecast_latest: {len(forecast_latest)} wierszy, analiza z prognozy: "
      f"{observed['time_utc'].min()} – {observed['time_utc'].max()}")
forecast.head()

{'issues': 52, 'first_issue': Timestamp('2026-10-13 09:00:00+0000', tz='UTC'), 'last_issue': Timestamp('2026-10-15 14:00:00+0000', tz='UTC'), 'lead_range_h': (-47, 71), 'soil_missing': 96, 'soil_imputed': 0, 'missing_issues': 2}


forecast_latest: 769 wierszy, analiza z prognozy: 2026-10-12 00:00:00+00:00 – 2026-10-15 14:00:00+00:00


,point,issued_at,valid_at,precip_mm,rain_mm,snowfall_cm,snow_depth_cm,temp_c,soil_moist_m3m3,lead_h,soil_is_imputed
0,katowice,2026-10-13 10:00:00+00:00,2026-10-12 00:00:00+00:00,0.0,0.0,0.0,0.0,8.9,0.256,-34,False
1,katowice,2026-10-13 10:00:00+00:00,2026-10-12 01:00:00+00:00,0.0,0.0,0.0,0.0,8.5,0.257,-33,False
2,katowice,2026-10-13 10:00:00+00:00,2026-10-12 02:00:00+00:00,0.0,0.0,0.0,0.0,8.2,0.256,-32,False
3,katowice,2026-10-13 10:00:00+00:00,2026-10-12 03:00:00+00:00,0.0,0.0,0.0,0.0,7.7,0.256,-31,False
4,katowice,2026-10-13 10:00:00+00:00,2026-10-12 04:00:00+00:00,0.0,0.0,0.0,0.0,7.5,0.256,-30,False


Synop IMGW (Kraków-Balice, Bielsko-Biała, Zakopane) służy tylko do kontroli prognoz – nie ma odpowiednika
w historii. Uwaga: `suma_opadu` to suma z okresu synoptycznego, a nie opad godzinowy – stąd nazwa `precip_sum_mm`.

In [15]:
synop, synop_info = live_meteo.process_synop()
print(synop_info)
synop.tail(3)

{'rows': 191, 'stations': ['Bielsko Biała', 'Kraków', 'Zakopane']}


,station_id,station,data_pomiaru,godzina_pomiaru,temp_c,wind_ms,wind_dir_deg,rel_humidity_pct,precip_sum_mm,pressure_hpa,fetched_at,time_utc
188,12625,Zakopane,2026-10-15,7,5.7,1,210,90.6,3.3,NaN,2026-10-15T08:01:13Z,2026-10-15 07:00:00+00:00
189,12625,Zakopane,2026-10-15,8,6.7,1,230,83.9,3.3,NaN,2026-10-15T09:01:15Z,2026-10-15 08:00:00+00:00
190,12625,Zakopane,2026-10-15,9,7.5,1,330,77.7,3.3,NaN,2026-10-15T10:01:13Z,2026-10-15 09:00:00+00:00


### 2.7 Ostrzeżenia IMGW – live

Czas ostrzeżeń jest w czasie polskim → UTC; `data_do = 9999-12-31` → brak końca (`NaT`).
`stopien = −1` to ostrzeżenie przed **suszą hydrologiczną** (flaga `is_drought`), nie odwołanie.

In [16]:
warnings, warnings_info = live_warnings.process()
print({k: v for k, v in warnings_info.items() if k != "flood_basin"})
warnings[warnings["dotyczy_zlewni"]][["opublikowano", "stopien", "zdarzenie", "data_od", "data_do", "is_drought"]].head()

{'rows': 96, 'basin': 27, 'drought': 94}


,opublikowano,stopien,zdarzenie,data_od,data_do,is_drought
15,2026-07-04 07:21:57+00:00,-1,Susza hydrologiczna,2026-07-04 07:22:43+00:00,NaT,True
16,2026-07-05 06:20:37+00:00,-1,Susza hydrologiczna,2026-07-05 06:21:23+00:00,NaT,True
19,2026-07-14 07:48:13+00:00,-1,Susza hydrologiczna,2026-07-14 08:00:00+00:00,NaT,True
21,2026-07-14 07:50:24+00:00,-1,Susza hydrologiczna,2026-07-14 08:00:00+00:00,NaT,True
24,2026-07-30 06:39:35+00:00,-1,Susza hydrologiczna,2026-07-30 06:40:23+00:00,NaT,True


### 2.8 Łączenie, tabela stacji, walidacja i zapis

Historia i live są przetwarzane osobno i tylko zestawiane – przez lukę w stanach wody (11.2025–10.2026)
**nigdy nie interpolujemy**; pochodzenie każdego wiersza mówi kolumna `source`.
Każda tabela dostaje schemat (`schema.py`: kolumny, typy, unikalny klucz). Jeśli jakakolwiek kontrola nie
przejdzie, tabele nie są zapisywane.

In [17]:
info = {"hist_hydro": hist_info, "live_hydro": live_info, "hist_meteo": era5_info,
        "forecast": forecast_info, "synop": synop_info, "warnings": warnings_info}
tables = build.assemble(hist_levels, live_hourly, era5, forecast, synop, warnings, info)

pd.DataFrame([{"tabela": name, "wiersze": len(df), "kolumny": df.shape[1],
               "klucz": ", ".join(schema.SCHEMAS[name]["key"]) if schema.SCHEMAS.get(name) else "date"}
              for name, df in tables.items()])

,tabela,wiersze,kolumny,klucz
0,stations,19,17,station_id
1,levels_daily,312341,9,"station_id, date"
2,levels_hourly,1330,9,"station_id, time_utc"
3,meteo_hourly,2871057,10,"point, time_utc"
4,meteo_daily,119637,12,"point, date"
5,forecast_archive,34848,11,"point, issued_at, valid_at"
6,forecast_latest,769,10,"point, valid_at"
7,synop_hourly,191,9,"station_id, time_utc"
8,warnings,96,16,"numer, biuro, opublikowano"
9,levels_daily_wide,16439,20,date


In [18]:
tables["stations"]

,station_id,name,river,downstream_id,river_km,lat,lon,gauge_zero_m,warning_cm,alarm_cm,record_cm,is_target,main_feature,note,hist_first,hist_last,live_last_utc
0,149180100,Skoczów,Wisła,149180210,NaN,NaN,NaN,NaN,210.0,260.0,356.0,False,True,<NA>,1950-11-01,2025-10-31,2026-10-15 14:20:00+00:00
1,149180210,Zabrzeg,Wisła,149180240,NaN,NaN,NaN,NaN,280.0,320.0,324.0,False,False,historia dopiero od 2010 r.,2010-11-01,2025-10-31,2026-10-15 14:20:00+00:00
2,149180240,Goczałkowice,Wisła,149190060,NaN,NaN,NaN,NaN,290.0,410.0,687.0,False,True,<NA>,1950-11-01,2025-10-31,2026-10-15 14:20:00+00:00
3,149190060,Jawiszowice,Wisła,150190170,NaN,NaN,NaN,NaN,480.0,630.0,825.0,False,True,<NA>,1950-11-01,2025-10-31,2026-10-15 14:20:00+00:00
4,149190100,Żywiec,Soła,149190120,NaN,NaN,NaN,NaN,280.0,340.0,393.0,False,True,<NA>,1955-11-01,2025-10-31,2026-10-15 14:20:00+00:00
5,149190120,Czaniec-Kobiernice,Soła,150190160,NaN,NaN,NaN,NaN,320.0,450.0,399.0,False,False,historia dopiero od 2013 r.,2013-11-01,2025-10-31,2026-10-15 14:20:00+00:00
6,149190170,Zator,Skawa,150190260,NaN,NaN,NaN,NaN,240.0,370.0,512.0,False,True,<NA>,1970-11-01,2025-10-31,2026-10-15 14:20:00+00:00
7,149190180,Wadowice,Skawa,149190170,NaN,NaN,NaN,NaN,370.0,490.0,398.0,False,True,<NA>,1950-11-01,2025-10-31,2026-10-15 14:20:00+00:00
8,149190200,Sucha Beskidzka,Stryszawka,149190210,NaN,NaN,NaN,NaN,220.0,270.0,385.0,False,False,od czerwca 2026 r. brak nowych odczytów live,1970-11-01,2025-10-31,NaT
9,149190210,Sucha Beskidzka,Skawa,149190180,NaN,NaN,NaN,NaN,215.0,285.0,473.0,False,True,<NA>,1950-11-01,2025-10-31,2026-10-15 14:20:00+00:00


In [19]:
problems = build.validate(tables)
print("Kontrole: OK" if not problems else "\n".join(problems))

Kontrole: OK


In [20]:
if not problems:
    report = build.save(tables, info)
    print(f"Zapisano {len(tables)} tabel w {config.PROCESSED.relative_to(ROOT)}")
    print(f"Raport jakości: {report.relative_to(ROOT)}")

Zapisano 10 tabel w data\processed
Raport jakości: reports\data_quality.md


### Podsumowanie preprocessingu

| Tabela | Zawartość |
|---|---|
| `levels_daily` | dobowe stany wody 19 stacji: historia 1980–2025 + live od 10.2026 (`source`) |
| `levels_daily_wide` | to samo w układzie data × stacja (`lvl_<station_id>`) |
| `levels_hourly` | godzinowe stany wody – tylko live |
| `meteo_hourly`, `meteo_daily` | pogoda w 7 punktach: ERA5 + analiza z prognozy dla ostatnich dni |
| `forecast_archive`, `forecast_latest` | prognozy pogody znane w danym momencie |
| `stations` | sieć rzeczna, progi, rekordy, `main_feature` |
| `synop_hourly`, `warnings` | kontrola prognoz i porównanie z ostrzeżeniami IMGW |

**Na co uważać w EDA i przy cechach:**
- luka w stanach wody 1.11.2025–11.10.2026 – nie łączyć szeregów przez tę przerwę,
- wartości `is_imputed` (uzupełnione) i `is_suspect` (podejrzane, m.in. skoki przy lodzie),
- definicja doby (`config.DAILY_DEFINITION`) to wciąż założenie – otwarta decyzja nr 1,
- kandydaci na zmianę zera wodowskazu (raport jakości) – sprawdzić na wykresach,
- Zabrzeg, Czaniec-Kobiernice i Stryszawka mają `main_feature = False`.

## 3. Eksploracyjna analiza danych (EDA)

Dane są czyste – teraz trzeba je zrozumieć. EDA ma odpowiedzieć na pytania, od których zależy projekt cech
i wybór modelu:

1. **Jak zachowuje się Wisła w Krakowie?** Rozkład stanów, sezonowość (roztopy wiosną, ulewy latem),
   częstość i czas trwania przekroczeń 370 i 520 cm, największe wezbrania (1997, 2001, 2010).
2. **Jak szybko woda płynie do Krakowa?** Opóźnienia między stacjami w górze rzeki a Bielanami
   (korelacje z przesunięciem) – to wyznaczy, które stacje są najważniejsze dla prognozy na 1, 2 i 3 dni.
3. **Jak opad przekłada się na wezbranie?** Związek sum opadu w zlewni z późniejszym wzrostem stanu,
   rola wilgotności gleby i topniejącego śniegu.
4. **Czy dane są stabilne w czasie?** Braki na stację i rok, kandydaci na zmianę zera wodowskazu,
   wpływ zbiorników (Świnna Poręba od ok. 2017 r.) – czy okres 1983–2014 jest dobrym zbiorem treningowym.
5. **Czym różni się okres live od historii?** Analiza z prognozy a ERA5, rozkłady stanów w październiku.

Dane do EDA wczytujemy z `data/processed/` – tabele zbudowane w sekcji 2.

In [21]:
from src.data import load_table

levels = load_table("levels_daily")
levels_wide = load_table("levels_daily_wide")
meteo_daily = load_table("meteo_daily")
stations_tab = load_table("stations")
missing_by_year = pd.read_csv(ROOT / "reports" / "missing_by_year.csv", index_col="station_id")

pd.DataFrame({
    "tabela": ["levels_daily", "levels_daily_wide", "meteo_daily", "stations"],
    "wiersze": [len(levels), len(levels_wide), len(meteo_daily), len(stations_tab)],
    "okres": [f"{levels['date'].min():%Y-%m-%d} – {levels['date'].max():%Y-%m-%d}",
              f"{levels_wide['date'].min():%Y-%m-%d} – {levels_wide['date'].max():%Y-%m-%d}",
              f"{meteo_daily['date'].min():%Y-%m-%d} – {meteo_daily['date'].max():%Y-%m-%d}", "–"],
})

,tabela,wiersze,okres
0,levels_daily,312341,1980-11-01 – 2026-10-15
1,levels_daily_wide,16439,1980-11-01 – 2026-10-15
2,meteo_daily,119637,1980-01-01 – 2026-10-16
3,stations,19,–
